<a href="https://colab.research.google.com/github/jason-snow58/Tuning-the-Stability-of-a-Disulfide-Stabilized-Phage-VLP-by-Interface-Guided-Capsid-Engineering/blob/main/Tuning_the_Stability_of_a_Disulfide_Stabilized_Phage_VLP_by_Interface_Guided_Capsid_Engineering_c_alpha_interface_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C-alpha interface map

Supporting analysis for *Tuning the Stability of a Disulfide-Stabilized Phage VLP by Interface-Guided Capsid Engineering*

---

### What this is

A flat, editable rendering of the residues the Figure 1 notebook classified: one circle per C-alpha, on two named layers, coloured by the classification. It opens as vector art rather than a raster image, so the residue classes can be styled without re-running anything.

This is a presentation tool for the interface classification. No map of this kind appears in the current figures — it is provided because it renders the published classification, not because it reproduces a published panel.

### The one thing that is not reproducible

**The projection follows the session camera.** Two viewpoints give two different maps of the same residues. The atom set, the layers and the colours are determined by the analysis; the geometry is a property of the view, so set the camera deliberately.

### Input

The `.pml` script written by the Figure 1 notebook — run that one first.

> **A note on structure.** Unlike the Figure 1 notebook, this step is a *preserved producer*: reading, deciding, drawing and writing happen inside one operation, and it changes PyMOL state directly. It is kept in that shape deliberately, because it reproduces the retained output and restructuring its internals would put that at risk. The phase separation described in the Figure 1 notebook does not apply here, and the notebook does not pretend otherwise.

## Setup

In [ ]:
#@title Setup and input controls { display-mode: "form" }
#@markdown Run this cell first. It detects the environment, installs anything
#@markdown missing, imports everything the notebook needs, and collects the
#@markdown input settings below. Defaults reproduce the published analysis.

# ---- environment -----------------------------------------------------------
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob
import os
import subprocess
import sys

def ensure(package, module=None):
    """Import a package, installing it first if this is Colab."""
    name = module or package
    try:
        __import__(name)
        return True
    except ImportError:
        if IN_COLAB:
            print(f"installing {package} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                           check=True)
            __import__(name)
            return True
        print(f"MISSING: {package}")
        print(f"  conda install -c conda-forge {package}")
        return False


# ---- dependencies ----------------------------------------------------------
# PyMOL is the dependency that differs most between the two environments: in
# Colab the open-source wheel installs cleanly, and locally it is usually
# already present in a conda environment.
READY = ensure("pymol-open-source", "pymol2")
import pymol2
print("PyMOL ready")

# ---- input controls --------------------------------------------------------
#@markdown **PyMOL script.** The `.pml` written by the Figure 1 notebook.
#@markdown Leave blank to search `output/` and `data/`.
PML_FILE = "" #@param {type:"string"}
#@markdown **Camera.** The projection follows the session camera, so this
#@markdown choice is part of the output. `orient` gives a reproducible view.
CAMERA = "orient" #@param ["orient", "as-loaded"]
#@markdown **Circle radius and canvas size, in SVG pixels.**
RADIUS_PX = 7.0 #@param {type:"number"}
WIDTH_PX = 1600 #@param {type:"integer"}
HEIGHT_PX = 1200 #@param {type:"integer"}
OUTPUT_DIR = "output" #@param {type:"string"}

# ---- helpers ---------------------------------------------------------------
def find_input(description, patterns, search_dirs, allow_multiple=False):
    """Locate an input file, or offer an upload box in Colab.

    Patterns are tried in order, so a specific name wins over a general glob.
    Returns one path unless allow_multiple=True.
    """
    for pattern in patterns:
        hits, seen = [], set()
        for directory in search_dirs:
            # '**' searches subdirectories, which matters because each analysis
            # is committed into its own directory named after the structure.
            found = glob.glob(os.path.join(directory, pattern), recursive=True)
            for path in sorted(found):
                real = os.path.realpath(path)
                if os.path.isfile(path) and real not in seen:
                    seen.add(real)
                    hits.append(path)
        if hits:
            print(f"input: matched {pattern!r}")
            for h in hits:
                print("   ", h)
            if len(hits) > 1 and not allow_multiple:
                print("   using the first; set the variable directly to choose another")
                return hits[0]
            return hits if allow_multiple else hits[0]

    if IN_COLAB:
        from google.colab import files
        print(f"Upload {description}:")
        uploaded = files.upload()
        if not uploaded:
            return None
        names = list(uploaded)
        return names if allow_multiple else names[0]

    print(f"Nothing found for {description}.")
    print("Searched:", ", ".join(search_dirs))
    print("Patterns:", ", ".join(repr(p) for p in patterns))
    return None


def require(value, what):
    if value is None:
        raise SystemExit(f"No input for {what}. See the message above.")
    return value


def deliver(paths):
    """Report finished files, and download them in Colab."""
    paths = [paths] if isinstance(paths, str) else list(paths)
    for p in paths:
        size = os.path.getsize(p) if os.path.exists(p) else 0
        print(f"   {p}  ({size:,} bytes)")
    if IN_COLAB:
        from google.colab import files
        for p in paths:
            files.download(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print()
print("Colab" if IN_COLAB else "local Jupyter", "| python", sys.version.split()[0])
print("output directory:", os.path.abspath(OUTPUT_DIR))

installing pymol-open-source ...
PyMOL ready

Colab | python 3.13.15
output directory: /content/output


## The analysis code

These cells write the analysis package into the working directory, so the notebook is self-contained and nothing has to be fetched. This is the same source that accompanies the manuscript — read it if you want to check the calculation, or run straight past it.

In [ ]:
os.makedirs("capsid", exist_ok=True)
print("package directory ready")

package directory ready


In [ ]:
%%writefile capsid/__init__.py
"""Interface and contact analysis for ssRNA phage capsids.

WHICH PIPELINES ARE SEPARATED, AND WHICH ARE NOT

Only the contact-classification path implements the phase separation in its
call graph:

    decide.analyse(pdb)                  reads the structure; classifies; once
    render.render_analysis(decision)     pure; produces every finished file
    validate.validate_rendering(...)     three derivations compared row by row
    effect.commit_artifacts(...)         transport only; cannot render

``effect`` does not import ``render`` and never receives an analysis record, so
no output path can decide or render once writing has begun. Files are staged
and committed as a set, so a failed run leaves the previous output untouched
rather than a directory holding half of one run and half of another.

The three PyMOL and figure modules -- ``pymol_contacts``, ``calpha_svg`` and
``network_map`` -- are a different shape. Each interleaves reading, deciding,
drawing and writing in a single operation, and each writes files or changes
PyMOL state directly. They are kept that way deliberately, because they
reproduce the retained outputs and restructuring their internals would put that
at risk. The guarantees above describe the classification path only.
"""

import importlib

__all__ = ['artifacts', 'calpha_svg', 'decide', 'effect', 'manifest',
           'network_map', 'palette', 'pymol_contacts', 'records', 'render',
           'validate']


def __getattr__(name):
    """Import submodules on first use.

    Importing the package must not require every optional dependency. A
    notebook that only classifies contacts needs neither matplotlib nor PyMOL,
    and should not be made to install them to say ``import capsid``.
    """
    if name in __all__:
        module = importlib.import_module(f'.{name}', __name__)
        globals()[name] = module
        return module
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')


def __dir__():
    return sorted(__all__)

Writing capsid/__init__.py


In [ ]:
%%writefile capsid/calpha_svg.py
"""Export the interface residues of a PyMOL view as an editable SVG.

    PyMOL> run capsid/calpha_svg.py
    PyMOL> export_interface_calpha_svg("out.svg")

Draws one circle per C-alpha atom in the ``inter_dimer_only`` and
``both_interfaces`` selections that the contact-classification PyMOL script
creates, projected from the session's current camera onto a flat canvas and
written on two named layers, so the result opens as editable art in a vector
graphics editor rather than as a raster image.

Circle colour is taken from the atom's colour in the session, so the map carries
the same residue classification as the structural view it was exported from.
Circles are drawn far-to-near, so nearer residues overlap further ones as they
do on screen.

THE PROJECTION FOLLOWS THE CURRENT CAMERA. Two exports from different viewpoints
place the same residues differently; the map is a view of the structure, not a
canonical layout. Set the view before exporting, and record it if the exact
projection needs to be reproduced.

Requires PyMOL. Tested with PyMOL 3.2 (open-source).
"""

import math
from xml.sax.saxutils import escape

INTER_LAYER = 'inter'
BOTH_LAYER = 'both'
INKSCAPE_NS = 'http://www.inkscape.org/namespaces/inkscape'


# --------------------------------------------------------------------------
# pure geometry -- no PyMOL
# --------------------------------------------------------------------------

def view_matrices(view):
    """Split PyMOL's 18-float get_view() tuple into its named parts."""
    import numpy as np
    return (np.array(view[0:9], dtype=float).reshape((3, 3)),   # rotation
            np.array(view[9:12], dtype=float),                  # camera position
            np.array(view[12:15], dtype=float),                 # origin of rotation
            float(view[15]), float(view[16]), float(view[17]))  # clip front/back, ortho


def project(xyz, rotation, position, origin, height_px, use_perspective=False,
            field_of_view=20.0):
    """Model coordinates -> 2D camera-plane coordinates, plus depth.

    Orthographic by default, matching PyMOL's default orthoscopic camera. A
    perspective divide would silently rescale the map relative to the session it
    is being compared against.
    """
    import numpy as np
    cam = (xyz - origin) @ rotation.T + position
    if use_perspective:
        focal = 0.5 * height_px / math.tan(math.radians(field_of_view) * 0.5)
        depth = np.maximum(1e-6, -cam[:, 2])
        return cam[:, 0] * (focal / depth), cam[:, 1] * (focal / depth), cam[:, 2]
    return cam[:, 0].copy(), cam[:, 1].copy(), cam[:, 2]


def fit_to_canvas(x, y, width_px, height_px, padding_px):
    """Scale to fill the canvas, preserving aspect, flipping y for SVG."""
    import numpy as np
    xmin, xmax = float(x.min()), float(x.max())
    ymin, ymax = float(y.min()), float(y.max())
    dx = max(1e-9, xmax - xmin)
    dy = max(1e-9, ymax - ymin)
    scale = min((width_px - 2 * padding_px) / dx,
                (height_px - 2 * padding_px) / dy)
    return (padding_px + (x - xmin) * scale,
            padding_px + (ymax - y) * scale)


def circle_element(atom_id, layer, label, cx, cy, radius, fill, stroke,
                   stroke_width, opacity):
    return (f'<circle id="{escape(atom_id)}" data-group="{layer}" '
            f'data-label="{escape(str(label)) if label else ""}" '
            f'cx="{cx:.3f}" cy="{cy:.3f}" r="{radius:.3f}" '
            f'fill="{fill}" fill-opacity="{opacity:.3f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.3f}" />\n')


def svg_document(layers, width_px, height_px):
    """Assemble the finished SVG from {layer name: [circle strings]}."""
    parts = ['<?xml version="1.0" encoding="UTF-8"?>\n',
             f'<svg xmlns="http://www.w3.org/2000/svg" '
             f'xmlns:inkscape="{INKSCAPE_NS}" '
             f'width="{width_px}px" height="{height_px}px" '
             f'viewBox="0 0 {width_px} {height_px}">\n']
    for name, circles in layers.items():
        parts.append(f'<g id="layer_{name}" inkscape:label="{name}">\n')
        parts.extend(circles)
        parts.append('</g>\n')
    parts.append('</svg>\n')
    return ''.join(parts)


def rgb_string(triple):
    r, g, b = triple
    return f'rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})'


# --------------------------------------------------------------------------
# PyMOL-facing
# --------------------------------------------------------------------------

def _read_selection(selection, state=1):
    """The one place this module asks PyMOL for anything."""
    import numpy as np
    from pymol import cmd

    xyz = cmd.get_coords(selection, state=state)
    if xyz is None or len(xyz) == 0:
        return None, None, None
    meta = []
    cmd.iterate_state(state, selection,
                      'meta.append((model, segi, chain, resi, resn, name, '
                      'elem, ID, color, label))', space={'meta': meta})
    if len(meta) != len(xyz):
        raise RuntimeError(f'metadata/coordinate mismatch for {selection}: '
                           f'{len(meta)} vs {len(xyz)}')
    rgb = np.array([cmd.get_color_tuple(m[8]) for m in meta], dtype=float)
    return xyz.astype(float), rgb, meta


def export_interface_calpha_svg(out_svg='calpha_interface.svg',
                                inter_sel='inter_dimer_only',
                                both_sel='both_interfaces',
                                state=1, width_px=1600, height_px=1200,
                                padding_px=40, use_perspective=False,
                                radius_px=7.0, stroke_px=1.0,
                                stroke_rgb=(0, 0, 0), opacity=1.0):
    """Write one SVG circle per C-alpha in the two interface selections."""
    import numpy as np
    from pymol import cmd

    groups = []
    for layer, selection in ((INTER_LAYER, f'({inter_sel} and name CA)'),
                             (BOTH_LAYER, f'({both_sel} and name CA)')):
        xyz, rgb, meta = _read_selection(selection, state=state)
        if xyz is not None and len(xyz):
            groups.append((layer, xyz, rgb, meta))

    if not groups:
        raise ValueError(f'No CA atoms in ({inter_sel}) or ({both_sel}). '
                         f'Run the Figure 1 .pml script first -- it is what '
                         f'creates these selections.')

    xyz = np.vstack([g[1] for g in groups])
    rgb = np.vstack([g[2] for g in groups])
    meta = [m for g in groups for m in g[3]]
    layer_of = [g[0] for g in groups for _ in g[3]]

    rotation, position, origin, _, _, ortho = view_matrices(cmd.get_view())
    fov = float(cmd.get_setting_float('field_of_view')) if use_perspective else 20.0
    x, y, depth = project(xyz, rotation, position, origin, height_px,
                          use_perspective, fov)
    X, Y = fit_to_canvas(x, y, width_px, height_px, padding_px)

    layers = {INTER_LAYER: [], BOTH_LAYER: []}
    stroke = rgb_string([c / 255 for c in stroke_rgb])
    for i in np.argsort(depth):            # far to near, so near draws on top
        model, segi, chain, resi, resn, name, _elem, atom_id, _color, label = meta[i]
        layers[layer_of[i]].append(circle_element(
            f'{model}:{segi}:{chain}:{resn}{resi}:{name}:ID{atom_id}',
            layer_of[i], label, X[i], Y[i], radius_px,
            rgb_string(rgb[i]), stroke, stroke_px, opacity))

    with open(out_svg, 'w', encoding='utf-8') as fh:
        fh.write(svg_document(layers, width_px, height_px))

    drawn = sum(len(v) for v in layers.values())
    if drawn != len(meta):
        raise AssertionError(f'{len(meta)} atoms read but {drawn} circles written')
    print(f'Wrote {out_svg}: {len(layers[INTER_LAYER])} inter + '
          f'{len(layers[BOTH_LAYER])} both C-alpha circles '
          f'(orthoscopic flag={ortho})')
    return out_svg


try:
    from pymol import cmd as _cmd
    _cmd.extend('export_interface_calpha_svg', export_interface_calpha_svg)
except ImportError:
    pass

Writing capsid/calpha_svg.py


In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")
import capsid
print("analysis package ready:", ", ".join(capsid.__all__))

analysis package ready: artifacts, calpha_svg, decide, effect, manifest, network_map, palette, pymol_contacts, records, render, validate


## Build the session and export

Runs the PyMOL script to recreate the classified session, then exports the map.

In [ ]:
from capsid import calpha_svg
from IPython.display import SVG, display

pml_path = globals().get("_found") or PML_FILE or find_input(
    "the .pml script from the Figure 1 notebook",
    ["**/*.pml", "*.pml"], ["output", "data", "."])
require(pml_path, "the .pml script")

out_svg = os.path.join(OUTPUT_DIR, "interface_map.svg")

with pymol2.PyMOL() as session:
    cmd = session.cmd
    sys.modules["pymol"].cmd = cmd
    cmd.do(f"@{pml_path}")
    cmd.sync()
    for name in ("inter_dimer_only", "both_interfaces"):
        print(f"{name:<22}{cmd.count_atoms(name + ' and name CA')} residues")
    if CAMERA == "orient":
        cmd.orient("structure")
    calpha_svg.export_interface_calpha_svg(
        out_svg, radius_px=RADIUS_PX, width_px=WIDTH_PX, height_px=HEIGHT_PX)

text = open(out_svg).read()
print(f"\ncircles written   : {text.count('<circle')}")
display(SVG(out_svg))
deliver(out_svg)

Upload the .pml script from the Figure 2 notebook:
